# PDF Parsing with Claude

Sends PDF documents directly to Claude via the Anthropic SDK (base64-encoded) for document analysis and key findings extraction.

In [0]:
%pip install Anthropic
%restart_python

In [ ]:
import anthropic
import base64
import httpx
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
DATABRICKS_TOKEN = w.config.token

# Path to PDF file in Databricks volume
pdf_path = "/Volumes/shm/default/raw_pdfs/cauli_wingz.pdf"

# Read and encode the PDF
with open(pdf_path, "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

In [ ]:
import anthropic
import base64
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_url = w.config.host
token = w.config.token

# Send to Claude using base64 encoding
client = anthropic.Anthropic(
    base_url=f"{workspace_url}/serving-endpoints/",
    api_key=token
)
message = client.messages.create(
    model='databricks-claude-sonnet-4',
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_data
                    }
                },
                {
                    "type": "text",
                    "text": "What are the key findings in this document?"
                }
            ]
        }
    ],
)

print(message.content)